In [1]:
!pip install bertopic
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 47.6 MB/s eta 0:00:00


In [406]:
import pandas as pd
from bertopic import BERTopic
from bertopic.backend import MultiModalBackend
from bertopic.representation import KeyBERTInspired
import re
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import random
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary

In [407]:
df = pd.read_csv("/content/vibe_coding_combined_translated.csv")

In [408]:
df = df[["full_text_translated", "image_url"]]

In [430]:
def preprocess_tweet(text):
    #lower case text
    text = text.lower()
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Remove user mentions
    text = re.sub(r'@\w+', '', text)
    # Remove hashtags
    text = re.sub(r'#\w+', '', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    #remove symbols
    text = re.sub(r'[^\w\s]', '', text)
    #remove numbers
    text = re.sub(r'\d+', '', text)
    #Replace all "vibe code" into vibecode
    text = re.sub(r'vibe code', 'vibecode', text)
    #Replace all "vibe coding" into vibecoding
    text = re.sub(r'vibe coding', 'vibecoding', text)
    #Relace all "vibe coded" into vibecoded
    text = re.sub(r'vibe coded', 'vibecoded', text)

    return text

In [431]:
df_preprocessed = df['full_text_translated'].fillna('').apply(preprocess_tweet)
df_preprocessed = pd.concat([df_preprocessed, df['image_url']], axis=1)

In [432]:
df_preprocessed = df_preprocessed[df_preprocessed['full_text_translated'].apply(lambda x: len(x.split()) >= 4)]
df_preprocessed.count()

,0
full_text_translated,20507
image_url,6493


In [465]:
docs = df_preprocessed['full_text_translated'].tolist()
images  = df_preprocessed['image_url'].tolist()
for i in range(len(images)):
    if pd.isna(images[i]):
        images[i] = None
random.shuffle(docs)
docs[:10]

['vibecoding a monitor to monitor the situation',
 'i track my coding time now before vibecoding  hoursday typing after vibecoding  minutes directing ai the other  hours thinking testing shipping productivity didnt change what i do changed do you track your ai vs manual coding time',
 'been waiting for someone to build this used to do mbse back in the dod where wed model out all the reqs etc theres been a missing middle layer to act as the source of truth to enable vibecoding of real software this looks like the right future to me',
 'its normal to have bugs even if god created a species like humans there would be bugs and diseases what about human vibecoding',
 'vibecoding web games  w sunday',
 '  out of  products die in silence not because theyre bad but because nobody knows they exist launches vibecode camp v a week sprint for builders who want to  validate their idea  build an mvp with ai no coding experience required  get their',
 'vibecoding app with qwen qwqb in a few lines of 

In [466]:
embedding_model = SentenceTransformer("all-mpnet-base-v2")

umap_model = UMAP(n_neighbors=10, n_components=5, min_dist=0.0)
hdbscan_model = HDBSCAN(min_cluster_size=15, min_samples=1)
vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_df=0.8,
    min_df=2
)

representation_model = KeyBERTInspired()

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    representation_model=representation_model,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [467]:
topics, probs = topic_model.fit_transform(docs)

In [468]:
topics = topic_model.reduce_outliers(docs, topics)
topic_model.reduce_topics(docs, nr_topics=20)
topics = topic_model.topics_
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,8135,-1_vibecoding vibecoding_programming_vibe_coding,"[vibecoding vibecoding, programming, vibe, cod...",[you can vibecode so fast now that a new patte...
1,0,8809,0_vibecoding just_vibecoding vibecoding_vibeco...,"[vibecoding just, vibecoding vibecoding, vibec...",[using cursor with claude for vibecoding mark...
2,1,1325,1_vibe marketing_vibecoding vibe_dapps_dapp,"[vibe marketing, vibecoding vibe, dapps, dapp,...",[codexero is a vibecoding ai dapp builder for ...
3,2,976,2_vibecoding games_vibecoding game_vibecoding ...,"[vibecoding games, vibecoding game, vibecoding...",[the xontos are going to dominate the vibecodi...
4,3,271,3_vibecoding day_vibecoding weekend_today vibe...,"[vibecoding day, vibecoding weekend, today vib...",[ready for the weekend or still in the grind i...
5,4,249,4_gemini vibecoding_vibecoding gemini_google g...,"[gemini vibecoding, vibecoding gemini, google ...",[holy shit just iterations gemini pro nailed...
6,5,128,5_telegram bot_vibecoding chatgpt_chatgpt vibe...,"[telegram bot, vibecoding chatgpt, chatgpt vib...",[we are live on vibecoding a telegram bot join...
7,6,97,6_vibecode chrome_vibecode landing_vibecoding ...,"[vibecode chrome, vibecode landing, vibecoding...",[day vibecoding has developed a product that ...
8,7,82,7_elon_musk_elon musk_trump,"[elon, musk, elon musk, trump, offended, propa...",[woke liberals are getting triggered by this e...
9,8,82,8_doctors_doctor_medical_healthcare,"[doctors, doctor, medical, healthcare, clinica...",[tech has seriously gotten the entire healthca...


In [469]:
topic_words = topic_model.get_topics()
topic_words = [[word for word, _ in topic_model.get_topic(topic_id)] for topic_id in topic_model.get_topics().keys() if topic_id != -1]

# Prepare docs for coherence model
tokenized_docs = [doc.split() for doc in docs]
dictionary = Dictionary(tokenized_docs)
corpus = [dictionary.doc2bow(text) for text in tokenized_docs]

# Get topic words
topic_words_raw = [[word for word, _ in topic_model.get_topic(topic_id)] for topic_id in topic_model.get_topics().keys() if topic_id != -1]

# Process topic_words_raw to get individual words that exist in our dictionary
processed_topic_words = []
for topic_list in topic_words_raw:
    current_topic_words = []
    for word_or_ngram in topic_list:
        # Split n-grams into individual words
        individual_words = word_or_ngram.split()
        for word in individual_words:
            # Only add words that are in our dictionary
            if word in dictionary.token2id:
                current_topic_words.append(word)
    # Only append non-empty topic lists
    if current_topic_words:
        processed_topic_words.append(current_topic_words)

# Calculate C_V coherence
coherence_model_cv = CoherenceModel(topics=processed_topic_words, texts=tokenized_docs, dictionary=dictionary, coherence='c_v')
coherence_cv = coherence_model_cv.get_coherence()
print(f"Coherence (C_V): {coherence_cv}")


# Calculate NPMI coherence
coherence_model_npmi = CoherenceModel(topics=processed_topic_words, texts=tokenized_docs, dictionary=dictionary, coherence='c_npmi')
coherence_npmi = coherence_model_npmi.get_coherence()
print(f"Coherence (NPMI): {coherence_npmi}")

Coherence (C_V): 0.44632376867736295
Coherence (NPMI): 0.106966429757117
